In [1]:
import os
import math
import numpy as np
import pandas as pd

import MDAnalysis as mda
from MDAnalysis.lib.distances import capped_distance, minimize_vectors

from pathlib import Path
from datetime import datetime

import os, platform, warnings

warnings.filterwarnings(
    "ignore",
    message=r"Failed to use notebook backend:.*",
    category=UserWarning,
)

import pyvista as pv

pv.set_jupyter_backend(None)

if platform.system() == "Linux":
    try:
        pv.start_xvfb(wait=0.1)
    except Exception as e:
        raise RuntimeError(
            "Xvfb could not be started. Install it (e.g., apt-get install xvfb "
            "or conda-forge xorg-x11-server-xvfb) and retry."
        ) from e


from PIL import Image

from math import ceil
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from reportlab.lib.units import cm

CSV_PATH = Path("rmoi_and_ap_by_run_total.csv")
OUT_ROOT = Path("pdf_reports_controls")

COHORT_DIR = {
    "bigger_box_1230": Path("bigger_box_1230"),
    "martini2": Path("martini2"),
}

COHORT_LABEL = {
    "bigger_box_1230": "Larger Box",
    "martini2": "MARTINI 2.2",
}

INCLUDE_TYPES = ["bigger_box_1230", "martini2"]
FORCE_RERENDER = True
OUT_ROOT.mkdir(parents=True, exist_ok=True)


/home/go46vuw/miniconda3/lib/python3.12/site-packages/pyvista/plotting/utilities/xvfb.py:48: PyVistaDeprecationWarning: This function is deprecated and will be removed in future version of PyVista. Use vtk-osmesa instead.
  warnings.warn(


In [2]:
df = pd.read_csv(CSV_PATH)

required = ["type", "target", "peptide", "run", "RMOI", "aggregation_propensity"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

# normalize string fields (handles " yes" etc.)
for c in ["type", "target", "peptide", "run"]:
    df[c] = df[c].astype(str).str.strip()

if "matches_morphology_visual" in df.columns:
    df["matches_morphology_visual"] = (
        df["matches_morphology_visual"].astype(str).str.strip().str.lower()
    )

# numeric fields
df["aggregation_propensity"] = pd.to_numeric(df["aggregation_propensity"], errors="coerce")
df["RMOI"] = pd.to_numeric(df["RMOI"], errors="coerce")

# filter to requested cohorts only
df = df[df["type"].isin(INCLUDE_TYPES)].copy()

In [3]:
def _run_sort_key(r):
    # run_1, run_2, ...
    try:
        return int(str(r).split("_")[-1])
    except Exception:
        return str(r)

RUNS = sorted(df["run"].dropna().unique().tolist(), key=_run_sort_key)
TARGETS = sorted(df["target"].dropna().unique().tolist())

print("Runs:", RUNS)
print("Targets:", TARGETS)
print("Cohorts:", INCLUDE_TYPES)

Runs: ['run_1', 'run_2', 'run_3']
Targets: ['fiber', 'sphere']
Cohorts: ['bigger_box_1230', 'martini2']


In [4]:
VIEW_1 = dict(degx=5, degy=100, degz=0,  order="zyx", pad=1.1)
VIEW_2 = dict(degx=5, degy=100, degz=90, order="zyx", pad=1.1)

RENDER_DPI_SCALE = 0.55
WINDOW_SIZE_PX = (2200, 2200)
TEAL = ['#E0F2F1', '#B2DFDB', '#4DB6AC', '#26A69A', '#00796B']
PAPER_GRAY = '#3F3F3F'
BB_COLOR = TEAL[4]
SC_COLOR = TEAL[3]

In [5]:
BB_POINT_SIZE = 8
SC_POINT_SIZE = 6

CUTOFF_NM = 1.8
MIN_PNG_BYTES = 2000

USE_FILESYSTEM = False

_PNG_MAGIC = b"\x89PNG\r\n\x1a\n"

def is_valid_png(path: Path, min_bytes=MIN_PNG_BYTES):
    try:
        path = Path(path)
        if not path.exists(): return False
        if path.stat().st_size < min_bytes: return False
        with path.open("rb") as f:
            return f.read(8) == _PNG_MAGIC
    except Exception:
        return False


def _residue_anchors(U):
    """One anchor per residue (BB if present; else residue COM). Units: Å."""
    anchors = []
    for res in U.residues:
        bb = res.atoms.select_atoms("name BB")
        anchors.append(bb.positions[0] if bb.n_atoms else res.atoms.center_of_mass())
    return np.asarray(anchors, dtype=float)

def _largest_residue_cluster_from_anchors(anchors_A, cutoff_nm, box):
    """
    anchors_A: (n,3) Å
    cutoff_nm: nm
    box: MDAnalysis ts.dimensions (Å + angles)
    """
    cutoff_A = cutoff_nm * 10.0
    pairs = capped_distance(
        anchors_A, anchors_A, cutoff_A,
        box=box, return_distances=False
    )

    n = len(anchors_A)
    parent = np.arange(n, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i, j in pairs:
        if i != j:
            union(int(i), int(j))

    comps = {}
    for i in range(n):
        r = find(i)
        comps.setdefault(r, []).append(i)

    largest = max(comps.values(), key=len) if comps else [0]
    return np.array(largest, dtype=int)

def center_main_cluster_inplace_on_current_frame(U, cutoff_nm=CUTOFF_NM, use_bb_for_com=True):
    """
    Mutates positions of U.atoms in the current timestep:
      1) make residues in largest cluster contiguous (min image)
      2) translate cluster COM to box center (min image)
      3) wrap into primary box
    """
    box = U.dimensions.copy()
    anchors_all = _residue_anchors(U)

    cluster_res_idx = _largest_residue_cluster_from_anchors(anchors_all, cutoff_nm, box)

    idx = np.concatenate([U.residues[i].atoms.indices for i in cluster_res_idx])
    cluster = U.atoms[idx]
    com_group = cluster.select_atoms("name BB") if use_bb_for_com else cluster

    # 1) make residues contiguous near cluster COM
    com = com_group.center_of_mass()
    for ri in cluster_res_idx:
        res = U.residues[int(ri)]
        disp = res.atoms.center_of_mass() - com
        shift = minimize_vectors(disp[None, :], box=box)[0] - disp
        res.atoms.positions += shift

    # 2) translate cluster COM to box center
    com2 = com_group.center_of_mass()
    box_center = 0.5 * box[:3]
    delta = minimize_vectors((com2 - box_center)[None, :], box=box)[0]
    U.atoms.positions -= delta

    # 3) wrap
    U.atoms.pack_into_box(box=box)

def cell_vectors_from_dimensions(dim):
    """
    MDAnalysis dimensions: [a,b,c,alpha,beta,gamma]
      a,b,c in Å; angles in degrees.
    Return 3 cell vectors (Å) in Cartesian coordinates.
    """
    a, b, c, alpha, beta, gamma = dim
    alpha = math.radians(alpha)
    beta  = math.radians(beta)
    gamma = math.radians(gamma)

    va = np.array([a, 0.0, 0.0], dtype=float)
    vb = np.array([b * math.cos(gamma), b * math.sin(gamma), 0.0], dtype=float)

    cx = c * math.cos(beta)
    cy = c * (math.cos(alpha) - math.cos(beta) * math.cos(gamma)) / max(math.sin(gamma), 1e-8)
    cz_sq = c*c - cx*cx - cy*cy
    cz = math.sqrt(max(cz_sq, 0.0))
    vc = np.array([cx, cy, cz], dtype=float)

    return va, vb, vc

def unitcell_edges(va, vb, vc):
    """Edges for the parallelepiped cell."""
    o = np.array([0.0, 0.0, 0.0])
    pts = [
        o, va, vb, va+vb,
        vc, va+vc, vb+vc, va+vb+vc
    ]
    edges_idx = [
        (0,1),(0,2),(1,3),(2,3),
        (4,5),(4,6),(5,7),(6,7),
        (0,4),(1,5),(2,6),(3,7)
    ]
    return [(pts[i], pts[j]) for i, j in edges_idx]

def quat_axis_angle(axis, deg):
    ax, ay, az = axis
    n = (ax*ax + ay*ay + az*az)**0.5 or 1.0
    ax, ay, az = ax/n, ay/n, az/n
    th = math.radians(deg) * 0.5
    s = math.sin(th)
    return [ax*s, ay*s, az*s, math.cos(th)]

def quat_euler(degx=0.0, degy=0.0, degz=0.0, order="zyx"):
    def qmul(q2, q1):
        x1,y1,z1,w1 = q1
        x2,y2,z2,w2 = q2
        return [
            w2*x1 + x2*w1 + y2*z1 - z2*y1,
            w2*y1 - x2*z1 + y2*w1 + z2*x1,
            w2*z1 + x2*y1 - y2*x1 + z2*w1,
            w2*w1 - x2*x1 - y2*y1 - z2*z1
        ]
    qx = quat_axis_angle([1,0,0], degx)
    qy = quat_axis_angle([0,1,0], degy)
    qz = quat_axis_angle([0,0,1], degz)
    q  = [0,0,0,1]
    for c in order.lower():
        q = qmul({'x': qx, 'y': qy, 'z': qz}[c], q)
    return q

def quat_to_rotmat(q):
    x,y,z,w = q
    xx, yy, zz = x*x, y*y, z*z
    xy, xz, yz = x*y, x*z, y*z
    wx, wy, wz = w*x, w*y, w*z
    return np.array([
        [1-2*(yy+zz), 2*(xy-wz),   2*(xz+wy)],
        [2*(xy+wz),   1-2*(xx+zz), 2*(yz-wx)],
        [2*(xz-wy),   2*(yz+wx),   1-2*(xx+yy)]
    ], dtype=float)


def render_two_groups_pyvista(bb_pos, sc_pos, dims, out_png: Path, view_params,
                             bb_color=BB_COLOR, sc_color=SC_COLOR, line_color=PAPER_GRAY,
                             bb_point_size=BB_POINT_SIZE, sc_point_size=SC_POINT_SIZE,
                             window_size_px=WINDOW_SIZE_PX, dpi_scale=RENDER_DPI_SCALE):
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)

    w, h = window_size_px
    w = int(w * dpi_scale)
    h = int(h * dpi_scale)

    pl = pv.Plotter(off_screen=False, window_size=(w, h))
    pl.set_background("white")
    pl.disable_anti_aliasing()
    
    if bb_pos is not None and len(bb_pos) > 0:
        bb = pv.PolyData(bb_pos)
        pl.add_mesh(bb, color=bb_color, opacity=0.98,
                    render_points_as_spheres=True, point_size=bb_point_size)

    if sc_pos is not None and len(sc_pos) > 0:
        sc = pv.PolyData(sc_pos)
        pl.add_mesh(sc, color=sc_color, opacity=0.9,
                    render_points_as_spheres=True, point_size=sc_point_size)

    va, vb, vc = cell_vectors_from_dimensions(dims)
    for p0, p1 in unitcell_edges(va, vb, vc):
        pl.add_mesh(pv.Line(p0, p1), color=line_color, line_width=2)

    pl.camera.parallel_projection = True

    if "cam_dir" in view_params:
        cam_dir = np.asarray(view_params["cam_dir"], float)
        cam_up  = np.asarray(view_params.get("cam_up", safe_up_for(cam_dir)), float)
    else:
        q = quat_euler(view_params["degx"], view_params["degy"], view_params["degz"], view_params["order"])
        R = quat_to_rotmat(q)
        cam_dir = R @ np.array([0.0, 0.0, 1.0])
        cam_up  = R @ np.array([0.0, 1.0, 0.0])


    box_center = 0.5 * (va + vb + vc)
    # Camera axes
    cam_dir = cam_dir / (np.linalg.norm(cam_dir) + 1e-12)
    cam_up  = cam_up  / (np.linalg.norm(cam_up)  + 1e-12)
    cam_right = np.cross(cam_dir, cam_up)
    cam_right = cam_right / (np.linalg.norm(cam_right) + 1e-12)

    # Focus on the aggregate (not the box)
    pts = []
    if bb_pos is not None and len(bb_pos) > 0: pts.append(bb_pos)
    if sc_pos is not None and len(sc_pos) > 0: pts.append(sc_pos)
    pts = np.vstack(pts) if len(pts) else np.zeros((1,3), dtype=float)

    focal = pts.mean(axis=0)

    # Orthographic scale: project onto camera plane and fit tightly
    rel = pts - focal
    x = rel @ cam_right
    y = rel @ cam_up
    half_span = float(max(np.max(np.abs(x)), np.max(np.abs(y)), 1.0))  # Å

    # pad factor: >1 means zoom OUT slightly; <1 means zoom IN
    pad = float(view_params.get("pad", 1.12))  # default: small margin
    pl.camera.parallel_scale = half_span * pad

    # Distance is irrelevant for ortho size, but keep it reasonable for clipping
    dist = 8.0 * half_span
    pl.camera.position = (focal + cam_dir * dist).tolist()
    pl.camera.focal_point = focal.tolist()
    pl.camera.up = cam_up.tolist()

    # Save without producing notebook output
    pl.show(screenshot=str(out_png), interactive=False, auto_close=True)

    if not is_valid_png(out_png):
        raise RuntimeError(f"Screenshot failed or invalid PNG: {out_png}")
    
def principal_axes(pts):
    """
    pts: (N,3) array in Å.
    Returns (axes, evals) where axes[:,0] is the longest-axis direction.
    """
    pts = np.asarray(pts, float)
    c = pts.mean(axis=0, keepdims=True)
    X = pts - c
    C = (X.T @ X) / max(len(X), 1)   # covariance-ish
    evals, evecs = np.linalg.eigh(C) # ascending
    order = np.argsort(evals)[::-1]  # descending
    evals = evals[order]
    evecs = evecs[:, order]          # columns are axes
    # normalize (just in case)
    evecs = evecs / (np.linalg.norm(evecs, axis=0, keepdims=True) + 1e-12)
    return evecs, evals

def safe_up_for(cam_dir, preferred=np.array([0.0, 1.0, 0.0])):
    cam_dir = np.asarray(cam_dir, float)
    cam_dir = cam_dir / (np.linalg.norm(cam_dir) + 1e-12)
    up = preferred.copy()
    # if nearly parallel, switch preferred
    if abs(np.dot(cam_dir, up)) > 0.92:
        up = np.array([1.0, 0.0, 0.0])
    # orthonormalize
    up = up - np.dot(up, cam_dir) * cam_dir
    up = up / (np.linalg.norm(up) + 1e-12)
    return up

def render_two_views_for_peptide(peptide_dir: Path, out_png1: Path, out_png2: Path, cutoff_nm=CUTOFF_NM):
    top = peptide_dir / TOP_NAME
    traj = peptide_dir / TRAJ_NAME
    if not top.exists() or not traj.exists():
        raise FileNotFoundError(f"Missing {TOP_NAME} or {TRAJ_NAME} in {peptide_dir}")

    u = mda.Universe(str(top), str(traj))
    u.trajectory[-1]  # last frame
    center_main_cluster_inplace_on_current_frame(u, cutoff_nm=cutoff_nm)

    dims = u.dimensions.copy()
    bb_pos = u.atoms.select_atoms("name BB").positions.copy()
    sc_pos = u.atoms.select_atoms("not name BB").positions.copy()

    pts = np.vstack([bb_pos, sc_pos]) if (len(bb_pos) and len(sc_pos)) else (bb_pos if len(bb_pos) else sc_pos)

    axes, evals = principal_axes(pts)
    a1 = axes[:, 0]
    a2 = axes[:, 1]
    a3 = axes[:, 2]

    anis = float(evals[0] / max(evals[1], 1e-12))

    if anis > 1.35:
        V1 = dict(cam_dir=a3, cam_up=safe_up_for(a3, preferred=a2), pad=VIEW_1.get("pad", 1.05))
        V2 = dict(cam_dir=a2, cam_up=safe_up_for(a2, preferred=a1), pad=VIEW_2.get("pad", 1.05))
    else:
        V1 = VIEW_1
        V2 = VIEW_2

    render_two_groups_pyvista(bb_pos, sc_pos, dims, out_png1, V1)
    render_two_groups_pyvista(bb_pos, sc_pos, dims, out_png2, V2)



def png_to_jpeg(png_path: Path, jpg_path: Path, quality=75, max_px=1400):
    """
    Convert to JPEG and optionally downscale so the largest side is max_px.
    """
    png_path = Path(png_path)
    jpg_path = Path(jpg_path)
    jpg_path.parent.mkdir(parents=True, exist_ok=True)

    im = Image.open(png_path).convert("RGB")
    w, h = im.size
    m = max(w, h)
    if m > max_px:
        scale = max_px / m
        im = im.resize((int(w*scale), int(h*scale)), Image.Resampling.LANCZOS)

    im.save(jpg_path, "JPEG", quality=quality, optimize=True, progressive=True)
    return jpg_path

def build_group_pdf_onepage(entries, out_pdf: Path, title: str, subtitle: str = "",
                            max_cols: int = 3,
                            margin_cm: float = 0.7,
                            gutter_cm: float = 0.28,
                            inner_gutter_cm: float = 0.18,
                            caption_fs: int = 8,
                            header_fs: int = 12,
                            subtitle_fs: int = 8):
    """
    Single-page portrait contact sheet:
      - grid of peptides (<= max_cols columns)
      - each tile: caption + two views
    """
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    page_w, page_h = A4
    page_h = page_h - 3*cm
    c = canvas.Canvas(str(out_pdf), pagesize=(page_w, page_h))

    margin = margin_cm * cm
    gutter = gutter_cm * cm
    inner_gutter = inner_gutter_cm * cm

    # Header height (kept compact)
    header_h = 0.8 * cm if (title or subtitle) else 0.0

    usable_w = page_w - 2 * margin
    usable_h = page_h - 2 * margin - header_h

    n = len(entries)
    if n == 0:
        c.setFont("Helvetica-Bold", header_fs)
        c.drawString(margin, page_h - margin, title or "Empty group")
        c.setFont("Helvetica", subtitle_fs)
        c.drawString(margin, page_h - margin - 0.8*cm, "No peptides found.")
        c.save()
        return

    # Fixed max 3 columns; choose cols based on count
    cols = min(max_cols, 3)
    if n <= 2: cols = n
    elif n <= 6: cols = min(2, cols)

    rows = int(ceil(n / cols))

    tile_w = (usable_w - (cols - 1) * gutter) / cols
    tile_h = (usable_h - (rows - 1) * gutter) / rows

    # Caption band
    cap_h = 0.55 * cm  # bigger than before

    # Header
    y_top = page_h - margin
    if header_h > 0:
        c.setFont("Helvetica-Bold", header_fs)
        c.drawString(margin, y_top, title)
        if subtitle:
            c.setFont("Helvetica", subtitle_fs)
            c.drawString(margin, y_top - 0.60*cm, subtitle)

    grid_top = page_h - margin - header_h

    def draw_placeholder(x, y, w, h, text="missing"):
        c.rect(x, y, w, h, stroke=1, fill=0)
        c.setFont("Helvetica", max(7, caption_fs - 1))
        c.drawString(x + 0.15*cm, y + h - 0.45*cm, text)

    # Sort so the grid is meaningful (AP desc, then RMOI desc)
    def sort_key(e):
        ap = e.get("ap"); rm = e.get("rmoi")
        apk = ap if ap is not None else -1e18
        rmk = rm if rm is not None else -1e18
        return (-apk, -rmk, str(e.get("peptide","")))

    entries_sorted = sorted(entries, key=sort_key)

    for idx, e in enumerate(entries_sorted):
        r = idx // cols
        col = idx % cols

        tx = margin + col * (tile_w + gutter)
        ty_top = grid_top - r * (tile_h + gutter)
        ty = ty_top - tile_h

        # Caption (larger)
        peptide = str(e.get("peptide", "NA"))
        ap = e.get("ap"); rm = e.get("rmoi")
        cap = f"{peptide} | AP: {(f'{ap:.3f}' if ap is not None else 'NA')} | RMOI: {(f'{rm:.3f}' if rm is not None else 'NA')}"
        c.setFont("Helvetica", caption_fs)
        c.drawString(tx + 0.05*cm, ty_top - 0.45*cm, cap[:120])

        # Image area (use as much of tile as possible)
        img_area_h = tile_h - cap_h
        img_w = (tile_w - inner_gutter) / 2.0
        img_h = img_area_h

        img_y = ty
        img1 = Path(e.get("img1_path") or "")
        img2 = Path(e.get("img2_path") or "")

        def draw_img(path: Path, x, y, w, h):
            if (not path.exists()) or (path.stat().st_size < MIN_PNG_BYTES):
                draw_placeholder(x, y, w, h, "render failed")
                return
            try:
                jpg_path = Path(str(path)).with_suffix(".jpg")
                if (not jpg_path.exists()) or jpg_path.stat().st_mtime < path.stat().st_mtime:
                    png_to_jpeg(path, jpg_path, quality=75, max_px=1200)
                ir = ImageReader(str(jpg_path))
                # Fill the allocated box (minimal letterboxing)
                c.drawImage(ir, x, y, width=w, height=h, preserveAspectRatio=True, anchor='c', mask="auto")
            except Exception:
                draw_placeholder(x, y, w, h, "bad image")

        draw_img(img1, tx,                    img_y, img_w, img_h)
        draw_img(img2, tx + img_w + inner_gutter, img_y, img_w, img_h)

        # Tile outline (optional but helps scanning)
        c.setLineWidth(0.4)
        c.rect(tx, ty, tile_w, tile_h, stroke=1, fill=0)

    c.save()

In [6]:
IMG_CACHE_DIRNAME = "_render_cache"  # keep same as your notebook
TOP_NAME = "peptide-cg.gro"
TRAJ_NAME = "trajout.xtc"

def peptide_dir_for(cohort_type: str, run: str, peptide: str) -> Path:
    # bigger_box_1230/run_x/peptide
    return COHORT_DIR[cohort_type] / run / peptide

def cache_paths(peptide_dir: Path):
    cache_dir = peptide_dir / IMG_CACHE_DIRNAME
    return cache_dir / "view_1.png", cache_dir / "view_2.png"

def ensure_two_views(peptide_dir: Path, force=False):
    img1, img2 = cache_paths(peptide_dir)

    if (not force) and is_valid_png(img1) and is_valid_png(img2):
        return img1, img2

    img1.parent.mkdir(parents=True, exist_ok=True)
    render_two_views_for_peptide(peptide_dir, img1, img2, cutoff_nm=CUTOFF_NM)
    return img1, img2

In [7]:
summary = []

for cohort_type in INCLUDE_TYPES:
    cohort_name = COHORT_LABEL.get(cohort_type, cohort_type)

    for run in RUNS:
        for target in TARGETS:
            sub = df[(df["type"] == cohort_type) & (df["run"] == run) & (df["target"] == target)].copy()
            peptides = sub["peptide"].dropna().astype(str).unique().tolist()

            print(f"\n=== {cohort_type} | {run} | {target} | peptides={len(peptides)} ===")

            entries = []
            for pep in peptides:
                pdir = peptide_dir_for(cohort_type, run, pep)

                row = sub.loc[sub["peptide"] == pep].iloc[0] if (sub["peptide"] == pep).any() else None
                ap = float(row["aggregation_propensity"]) if (row is not None and pd.notna(row["aggregation_propensity"])) else None
                rm = float(row["RMOI"]) if (row is not None and pd.notna(row["RMOI"])) else None

                img1 = img2 = None
                try:
                    if pdir.exists():
                        # Optional: verify required files exist
                        if not (pdir / TOP_NAME).exists() or not (pdir / TRAJ_NAME).exists():
                            raise FileNotFoundError(f"Missing {TOP_NAME} or {TRAJ_NAME} in {pdir}")

                        img1, img2 = ensure_two_views(pdir, force=FORCE_RERENDER)
                    else:
                        print(f"  [WARN] Missing directory: {pdir}")
                except Exception as ex:
                    print(f"  [WARN] Render failed for {pep} ({pdir}): {ex}")

                entries.append(dict(
                    peptide=pep,
                    ap=ap,
                    rmoi=rm,
                    img1_path=str(img1) if img1 else None,
                    img2_path=str(img2) if img2 else None,
                ))

            out_pdf = OUT_ROOT / cohort_type / run / f"{target}.pdf"

            build_group_pdf_onepage(
                entries=entries,
                out_pdf=out_pdf,
                title=f"{cohort_name} | {run} | {target}",
                max_cols=3,
                caption_fs=8,
                header_fs=12,
                subtitle_fs=8,
            )

            summary.append((cohort_type, run, target, len(entries), str(out_pdf)))

print("\nDone. PDFs written:")
for cohort_type, run, target, n, path in summary:
    print(f"  {cohort_type:<14} | {run:<5} | {target:<6} | peptides={n:4d} | {path}")



=== bigger_box_1230 | run_1 | fiber | peptides=8 ===



=== bigger_box_1230 | run_1 | sphere | peptides=8 ===

=== bigger_box_1230 | run_2 | fiber | peptides=8 ===

=== bigger_box_1230 | run_2 | sphere | peptides=8 ===

=== bigger_box_1230 | run_3 | fiber | peptides=8 ===

=== bigger_box_1230 | run_3 | sphere | peptides=8 ===

=== martini2 | run_1 | fiber | peptides=8 ===

=== martini2 | run_1 | sphere | peptides=8 ===

=== martini2 | run_2 | fiber | peptides=8 ===

=== martini2 | run_2 | sphere | peptides=8 ===

=== martini2 | run_3 | fiber | peptides=8 ===

=== martini2 | run_3 | sphere | peptides=8 ===

Done. PDFs written:
  bigger_box_1230 | run_1 | fiber  | peptides=   8 | pdf_reports_controls/bigger_box_1230/run_1/fiber.pdf
  bigger_box_1230 | run_1 | sphere | peptides=   8 | pdf_reports_controls/bigger_box_1230/run_1/sphere.pdf
  bigger_box_1230 | run_2 | fiber  | peptides=   8 | pdf_reports_controls/bigger_box_1230/run_2/fiber.pdf
  bigger_box_1230 | run_2 | sphere | peptides=   8 | pdf_reports_controls/bigger_box_1230/run_2/sphere